# DantinoX Quickstart

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/winstonsmith1897/DantinoX/blob/main/docs/notebooks/01_quickstart.ipynb)

This notebook demonstrates the Level-1 (one-liner) and Level-2 APIs.
Runtime: GPU (T4) recommended.

In [ ]:
# Install DantinoX (skip if already installed)
!pip install -q git+https://github.com/winstonsmith1897/DantinoX.git#egg=dantinox[all]

In [ ]:
import jax
print('JAX version:', jax.__version__)
print('Devices:', jax.devices())

## Level 1 — One-liner API

Train an AR model in a single call:

In [ ]:
import dantinox as dx

# Download a small corpus for demo
import urllib.request, os
if not os.path.exists('tiny_shakespeare.txt'):
    url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
    urllib.request.urlretrieve(url, 'tiny_shakespeare.txt')
    print('Downloaded tiny_shakespeare.txt')

run_dir = dx.fit(
    'ar',
    'tiny_shakespeare.txt',
    dim=256, n_heads=4, head_size=64, num_blocks=4,
    vocab_size=200,
    lr=3e-4, epochs=2,
)
print('Checkpoint saved to:', run_dir)

In [ ]:
# Generate text from the trained checkpoint
output = dx.quick_generate(run_dir, 'HAMLET:\n', max_new_tokens=200)
print(output)

## Level 2 — Explicit paradigm

More control: explicit `ARParadigm`, `Trainer`, and `ModelConfig`:

In [ ]:
paradigm = dx.ARParadigm(
    dx.ModelConfig(dim=256, n_heads=4, head_size=64, num_blocks=4,
                   vocab_size=200, causal=True)
)
trainer = dx.Trainer(paradigm, dx.TrainingConfig(lr=3e-4, epochs=1, batch_size=16))
run_dir2 = trainer.fit('tiny_shakespeare.txt')
print('Run dir:', run_dir2)

## Quick FLOPs estimate

No model instance needed:

In [ ]:
flops = dx.profile(
    dx.ModelConfig(dim=256, n_heads=4, head_size=64, num_blocks=4, vocab_size=200),
    seq_len=256, batch_size=4
)
print(flops)